In [0]:
%run /Users/malkeet21.uk@gmail.com/Databricks-Data-Engineering-Assesment/common_utils/utils

#Task 4: Data analysis

In [0]:
from pyspark.sql.functions import col

# Step 1: read the three silver tables and convert 90_and_over from text ("350,700") to integer (350700)
df_persons = spark.table(SILVER_PERSONS_TABLE).withColumn("90_and_over", to_int("90_and_over"))
df_males = spark.table(SILVER_MALES_TABLE).withColumn("90_and_over", to_int("90_and_over"))
df_females = spark.table(SILVER_FEMALES_TABLE).withColumn("90_and_over", to_int("90_and_over"))

# Step 2: join Females and Males to Persons on year
# (rename each count first, because all three DataFrames have a column called 90_and_over)
persons = df_persons.select("year", col("90_and_over").alias("90_and_over_num_persons"))
females = df_females.select("year", col("90_and_over").alias("90_and_over_num_females"))
males = df_males.select("year", col("90_and_over").alias("90_and_over_num_males"))

df_joined = persons.join(females, on="year", how="inner").join(males, on="year", how="inner")
check_row_count(df_joined, YEARS_PER_POPULATION, "Joined")   # no year lost in the join

# Step 3: percentage split of 90_and_over between females and males, to 2 decimal places
df_split = (
    df_joined
    .withColumn("90_and_over_pcnt_females", percentage("90_and_over_num_females", "90_and_over_num_persons"))
    .withColumn("90_and_over_pcnt_males", percentage("90_and_over_num_males", "90_and_over_num_persons"))
)

# Step 4: display in the required column order, percentages shown as e.g. 76.87%
df_result = df_split.select(
    "year",
    "90_and_over_num_persons",
    as_percent_text("90_and_over_pcnt_females"),
    "90_and_over_num_females",
    as_percent_text("90_and_over_pcnt_males"),
    "90_and_over_num_males",
).orderBy("year")

display(df_result)

#Task 5: Persist and query 
1.Write the Task 4 result to a Delta table in Unity Catalog, in the catalog and schema your interviewer gives you. Call the table very_old_90_over_split.


In [0]:
write_delta_table(df_gold,very_old_90_over_split)